# 4. Evaluate the remaining operator insertions

Restore the full singlet coordinate system, evaluate additional operator monomials, and export enriched model records. See the [README](README.md) and [data guide](docs/DATA_AND_LIMITATIONS.md).

**Execution status:** `stable_models.json` and its producer, `saving_models.py`, are missing. The fitting notebook does not directly produce this input. Printed values are singlet monomial factors; they are not complete physical observables or an experimental exclusion test.

## Imports and matching scan settings

Use the same retained-monomial limit as the fit that produced the model records.

In [ ]:
import json
from copy import deepcopy
import numpy as np
n_o1_coef_per_entry = 2


## Load stabilized records and single-Higgs libraries

The operator library must contain all three indexed families. `stable_models.json` must use the positional record layout described in the data guide.

In [ ]:
with open('stable_models.json') as file:
    stable_models = json.load(file)


In [ ]:
with open('R_parity_library2.json') as file:
    R_parity_library = json.load(file)


In [ ]:
with open('mu_term_library2.json') as file:
    mu_term_library = json.load(file)


In [ ]:
with open('library_ALL_v4.json', 'r') as file:
    library = json.load(file)


In [ ]:
with open('insertions_matrices.json', 'r') as file:
    matrices = json.load(file)


## Recover full singlet coordinates

The code rebuilds retained insertion lists from the original matrices. It assigns zero to both deliberately killed VEVs and coordinates unused by the fit. An unused VEV is not determined physically by the fit; the reported suppression depends on this additional zero assignment.

In [ ]:
def retrieve_original_insertions_matrices(matrix_up, matrix_down, vevs_to_be_killed):
    """Recover retained monomials in the full singlet-coordinate basis.

    Uses the configured truncation and notebook-level singlet counts."""
    original_up_matrix = [[[], [], []], [[], [], []], [[], [], []]]
    original_down_matrix = [[[], [], []], [[], [], []], [[], [], []]]
    for i in range(3):
        for j in range(3):
            for k in range(len(matrix_up[i][j])):
                check = any((matrix_up[i][j][k][l] != 0 for l in vevs_to_be_killed))
                if check == False and len(original_up_matrix[i][j]) < n_o1_coef_per_entry:
                    original_up_matrix[i][j].append(matrix_up[i][j][k])
            for k in range(len(matrix_down[i][j])):
                check = any((matrix_down[i][j][k][l] != 0 for l in vevs_to_be_killed))
                if check == False and len(original_down_matrix[i][j]) < n_o1_coef_per_entry:
                    original_down_matrix[i][j].append(matrix_down[i][j][k])
    field_usage = np.zeros(shape=n_of_phi)
    for i in range(3):
        for j in range(3):
            if len(original_up_matrix[i][j]) != 0:
                for k in range(len(original_up_matrix[i][j])):
                    field_usage += np.array(original_up_matrix[i][j][k])
            if len(original_down_matrix[i][j]) != 0:
                for k in range(len(original_down_matrix[i][j])):
                    field_usage += np.array(original_down_matrix[i][j][k])
    for i in range(n_of_phi):
        if field_usage[i] != 0:
            if i < n_of_non_pert_phi:
                field_usage[i] = -1
            else:
                field_usage[i] = 1
    return (field_usage, original_up_matrix, original_down_matrix)


## Enrich the legacy positional records

Start from a fresh copy of `stable_models` each time this cell runs, restore the full coordinates, and append the operator and mu libraries.

In [ ]:
def identifier(s):
    """Remove the final comma-separated restart suffix from a legacy fit key."""
    return s.rsplit(',', 1)[0].strip()

def insert_zeros(data_list, reference_list):
    """Restore omitted singlet slots as zeros and preserve the parameter tail.

    This legacy convention also sets unfitted singlet VEVs to zero."""
    result = []
    data_index = 0
    for value in reference_list:
        if value == 0:
            result.append(0)
        elif data_index < len(data_list):
            result.append(data_list[data_index])
            data_index += 1
    result.extend(data_list[data_index:])
    return result
models = deepcopy(stable_models)
for i in models:
    id = identifier(i)
    n_of_non_pert_phi = len(library[id][0])
    n_of_pert_phi = len(library[id][1])
    n_of_phi = n_of_non_pert_phi + n_of_pert_phi
    sacrified_vevs = models[i][-1]
    pattern, original_up, original_down = retrieve_original_insertions_matrices(matrices[0][id], matrices[1][id], sacrified_vevs)
    models[i][3] = insert_zeros(models[i][3], pattern)
    models[i][2][0] = original_up
    models[i][2][1] = original_down
    models[i].append(R_parity_library[id])
    models[i].append(mu_term_library[id])
    models[i].append([matrices[0][id], matrices[1][id]])


## Evaluate HL monomials

Each insertion contributes $\prod_a v_a^{p_a}$. The following cells print nonzero monomials individually, without combining unknown operator coefficients.

In [ ]:
print('############################## Couplings of the type -----> rho_p Hbar L^p ############################## \n \n')
for i in models:
    print('########## MODEL ID:', i, '###########\n')
    rho = models[i][10][0]
    for j in range(3):
        for k in range(len(rho[j])):
            insertion = 1
            for l in range(len(rho[j][k])):
                if rho[j][k][l] != 0:
                    insertion *= models[i][3][l] ** rho[j][k][l]
            if insertion != 0:
                print('rho_', j, ' = ', insertion, rho[j][k])


## Evaluate FFT monomials

In [ ]:
print('#################################  Couplings of the type -----> lambda_(p,q,r)  5^p 5^q 10^r ################################# \n\n')
for i in models:
    print('\n\n########## MODEL ID:', i, '###########\n')
    llambda = models[i][10][1]
    for j in range(3):
        for k in range(3):
            for l in range(3):
                for m in range(len(llambda[j][k][l])):
                    insertion = 1
                    for n in range(len(llambda[j][k][l][m])):
                        if llambda[j][k][l][m][n] != 0:
                            insertion *= models[i][3][n] ** llambda[j][k][l][m][n]
                    if insertion != 0:
                        print('lambda_', j, k, l, ' = ', insertion)


## Evaluate FTTT monomials

In [ ]:
print("######################## Couplings of the type -----> lambda'_(p,q,r,s)  5^p 10^q 10^r 10^s ######################## \n\n")
for i in models:
    print('\n\n########## MODEL ID:', i, '###########\n')
    llambda = models[i][10][2]
    for j in range(3):
        for k in range(3):
            for l in range(3):
                for lp in range(3):
                    for m in range(len(llambda[j][k][l][lp])):
                        insertion = 1
                        for n in range(len(llambda[j][k][l][lp][m])):
                            if llambda[j][k][l][lp][m][n] != 0:
                                insertion *= models[i][3][n] ** llambda[j][k][l][lp][m][n]
                        if insertion != 0:
                            print('lambda_', j, k, l, lp, ' = ', insertion)


## Inspect surviving mu monomials

Print every mu monomial that remains nonzero after coordinate restoration. No output beneath a model heading means all enumerated terms vanish at these VEVs.

In [ ]:
for model_key, record in models.items():
    model_id = identifier(model_key)
    print(f'Model {model_key}: surviving mu monomials')
    for exponent in mu_term_library[model_id]:
        factor = np.prod([record[3][index] ** power for index, power in enumerate(exponent)])
        if factor != 0:
            print(exponent, '->', factor)


## Export enriched records

`final_models.json` contains the enriched legacy records. Printed monomial values are not separately collected into this file.

In [ ]:
with open('final_models.json', 'w') as outfile:
    json.dump(models, outfile)
